# ChatBot

## Check which python

In [2]:
import sys
print(sys.executable)

d:\Code\swe_agent_jom\.venv\Scripts\python.exe


## load deepseek api

In [3]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path)

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", deepseek_api_key is not None)

DEEPSEEK_API_KEY loaded: True


## LLM API Call Example

In [4]:
from openai import OpenAI

client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com",
)

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {
            "role": "system",
            "content": "You are a research helper. Be accurate and concise"
        },
        {
            "role": "user",
            "content": "Explain what is AI Agent in 3 key points."
        }
    ],
    stream=False,
)

print(response.choices[0].message.content)

1. **Autonomous Decision-Making**: An AI agent operates independently, interpreting its environment and taking actions without constant human intervention.
2. **Perception and Action**: It uses sensors (data inputs) to perceive its surroundings and actuators (tools or APIs) to execute actions, forming a continuous loop of observation and response.
3. **Goal-Oriented Behavior**: The agent is designed to achieve specific objectives, often optimizing its actions through reasoning or learning (e.g., reinforcement learning).


## Stream output & temperature

In [5]:
stream  = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {
            "role": "system",
            "content": "You are a rigorous research assistant. Your responses should be well-structured, verifiable, and free from fabricated information."
        },
        {
            "role": "user",
            "content": "Introduce Agent Memory to me with 3 key point."
        }
    ],
    stream=True,
    temperature=0.1 # Smaller lead to more stable and comservative, larger lead to more creative and emanative
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="")

Here are three key points about **Agent Memory** in the context of AI agents (e.g., LLM-based agents):

1. **Types of Memory (Short-Term vs. Long-Term)**  
   Agent memory is often categorized into two main forms:  
   - *Short-term / working memory*: Holds information from the immediate conversation or task (e.g., the current chat history). It is typically limited in capacity and volatile.  
   - *Long-term memory*: Stores persistent knowledge, past interactions, or learned patterns across sessions. This can be implemented via external databases, vector stores, or compressed summaries.

2. **Mechanisms for Storage & Retrieval**  
   To make memory useful, agents employ techniques such as:  
   - **Embedding-based retrieval**: Observations or interactions are converted into vector embeddings and stored in a vector database. When needed, the agent retrieves semantically similar memories (e.g., using cosine similarity).  
   - **Compression & summarization**: Long histories are periodica

## Structured output

In [6]:
client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com",
)

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {"role": "system", "content": "You are a research helper. Be accurate and concise. You can only output valid json object."},
        {"role": "user", "content": "Explain what is AI Agent in 3 key points."}
    ],
    stream=False,
    response_format={"type": "json_object"},
)
print(response.choices[0].message.content)

{
  "key_points": [
    "An AI agent is an autonomous entity that perceives its environment using sensors and acts upon that environment using actuators to achieve specific goals.",
    "It typically incorporates a decision-making process, often using machine learning or rule-based systems, to choose actions based on its observations and internal state.",
    "AI agents can be categorized into types such as reactive agents (no memory), deliberative agents (with internal models), and learning agents (improve over time)."
  ]
}


## Multiple rounds of conversation

In [7]:
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

MODEL = "deepseek-v4-flash"  

messages: list[ChatCompletionMessageParam] = [
    {
        "role": "system",
        "content": "You are a rigorous research assistant. Be consice and accurate."
    }
]

def chat(user_input: str) -> str|None:
    messages.append({
        "role": "user",
        "content": user_input
    })

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2,
    )

    assistant_reply = response.choices[0].message.content

    messages.append({
        "role": "assistant",
        "content": assistant_reply
    })

    return assistant_reply

In [8]:
chat("My name is JOM, I am working on AI Agents.")

"Hello JOM. I can assist with your work on AI Agents—whether it's architecture, reasoning, tool use, or related research. What specific aspect are you focusing on?"

In [9]:
chat("What is my name? And what I am focusing on?")

'Your name is JOM, and you are focusing on AI Agents.'

In [10]:
chat("How do you know that?")

'I know that because you stated it in your first message: "My name is JOM, I am working on AI Agents." I simply retained that information from our conversation.'

In [11]:
chat("What if the message is too large?")

"In conversational AI systems, if a message (or the cumulative context) exceeds the model's maximum token limit, a few things can happen:\n\n1. **Truncation**: The oldest parts of the conversation are cut off to make room for the new input. This means earlier information—like your name and focus—may be lost.\n2. **Summarization**: Some systems insert a compressed summary of earlier turns to preserve key facts within the context window.\n3. **Chunking/Retrieval**: In agent architectures, long contexts are often split into chunks and stored externally (e.g., vector database), retrieved only when relevant.\n\nIn this specific session, I have no internal memory beyond the current conversation. If the total token count (your messages + my replies) exceeded my model's limit, I would lose the earlier mention of your name and focus. For research on AI Agents, this is a critical design consideration—managing context windows effectively is essential for long-running, coherent agent interactions.

In [12]:
for message in messages:
    role = message["role"]
    content = message.get("content", "")
    print(f"{role}: {content}")
    print()

system: You are a rigorous research assistant. Be consice and accurate.

user: My name is JOM, I am working on AI Agents.

assistant: Hello JOM. I can assist with your work on AI Agents—whether it's architecture, reasoning, tool use, or related research. What specific aspect are you focusing on?

user: What is my name? And what I am focusing on?

assistant: Your name is JOM, and you are focusing on AI Agents.

user: How do you know that?

assistant: I know that because you stated it in your first message: "My name is JOM, I am working on AI Agents." I simply retained that information from our conversation.

user: What if the message is too large?

assistant: In conversational AI systems, if a message (or the cumulative context) exceeds the model's maximum token limit, a few things can happen:

1. **Truncation**: The oldest parts of the conversation are cut off to make room for the new input. This means earlier information—like your name and focus—may be lost.
2. **Summarization**: Some

In [13]:
print(messages)

[{'role': 'system', 'content': 'You are a rigorous research assistant. Be consice and accurate.'}, {'role': 'user', 'content': 'My name is JOM, I am working on AI Agents.'}, {'role': 'assistant', 'content': "Hello JOM. I can assist with your work on AI Agents—whether it's architecture, reasoning, tool use, or related research. What specific aspect are you focusing on?"}, {'role': 'user', 'content': 'What is my name? And what I am focusing on?'}, {'role': 'assistant', 'content': 'Your name is JOM, and you are focusing on AI Agents.'}, {'role': 'user', 'content': 'How do you know that?'}, {'role': 'assistant', 'content': 'I know that because you stated it in your first message: "My name is JOM, I am working on AI Agents." I simply retained that information from our conversation.'}, {'role': 'user', 'content': 'What if the message is too large?'}, {'role': 'assistant', 'content': "In conversational AI systems, if a message (or the cumulative context) exceeds the model's maximum token limi